**Linda Zier**

**ST 554**

**HW #9**

## The Goal

• Finding a data set you can fit supervised learning models with

• Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.

• Writing a narrative (via a notebook) with explanations and discussions as you go through the above.

## The Data

The dataset I used gives insights into bike rentals in Seoul Korea based on environmental factors.  I will use these as well as day of week and month considerations to model number of bikes rented.


**Setup**

First I have to setup everything. This includes installing pyspark, importing, and starting  my Spark Session.

In [1]:
# committing regularly
#!cd ST-554-repo && git add -A && git commit -m "progress" && git push

In [2]:
# clone my hub - just each first time I get started
#!git clone https://github.com/ljzier/ST-554-repo.git
!pip install pyspark


Defaulting to user installation because normal site-packages is not writeable


In [3]:

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, SQLTransformer

from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GeneralizedLinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print("imports ran")

imports ran


In [4]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 08:23:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# read in the data and inspect
import pandas as pd

df = pd.read_csv('ST-554-repo/data/SeoulBikeData.csv', encoding='latin1')
print(df.head())
print(df.dtypes)
print(df.shape)


         Date  Rented Bike Count  Hour  Temperature(°C)  Humidity(%)  \
0  01/12/2017                254     0             -5.2           37   
1  01/12/2017                204     1             -5.5           38   
2  01/12/2017                173     2             -6.0           39   
3  01/12/2017                107     3             -6.2           40   
4  01/12/2017                 78     4             -6.0           36   

   Wind speed (m/s)  Visibility (10m)  Dew point temperature(°C)  \
0               2.2              2000                      -17.6   
1               0.8              2000                      -17.6   
2               1.0              2000                      -17.7   
3               0.9              2000                      -17.6   
4               2.3              2000                      -18.6   

   Solar Radiation (MJ/m2)  Rainfall(mm)  Snowfall (cm) Seasons     Holiday  \
0                      0.0           0.0            0.0  Winter  No Holiday   


(Note to self: I'll get an error if I try to run this again because I renamed my columns, FYI.  Only run it once at the start)

In [6]:

#see if there are any non-functioning days
print(df['Functioning Day'].value_counts())
print(df[df['Functioning Day'] == 'No']['Rented Bike Count'].unique())

#filter out the non-functioning days and check the number of rows
df = df[df['Functioning Day'] == 'Yes']
print(df.shape)

Functioning Day
Yes    8465
No      295
Name: count, dtype: int64
[0]
(8465, 14)


**Clean and Convert**

Since I already loaded my data, now I clean and convert my dataframe to a Sparkdata frame.

In [7]:


#rename columns to make the names spark-friendly (no spaces)
df.columns = [
    'Date', 'Bike_Count', 'Hour', 'Temperature', 'Humidity',
    'Wind_Speed', 'Visibility', 'Dew_Point', 'Solar_Radiation',
    'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day'
]

#convert to Spark data frame
sdf= spark.createDataFrame(df)

# month as new feature
sdf = sdf.withColumn('Month', F.month(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# day of week as new feature
sdf = sdf.withColumn('Day_of_Week', F.dayofweek(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# make the type DoubleType which is like Float64
numeric_cols = ['Bike_Count', 'Hour', 'Temperature', 'Humidity', 'Wind_Speed', 
                'Visibility', 'Dew_Point', 'Solar_Radiation', 'Rainfall', 
                'Snowfall', 'Month', 'Day_of_Week']

for c in numeric_cols:
    sdf = sdf.withColumn(c, F.col(c).cast(DoubleType()))

sdf.printSchema()
sdf.show(5)

root
 |-- Date: string (nullable = true)
 |-- Bike_Count: double (nullable = true)
 |-- Hour: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- Visibility: double (nullable = true)
 |-- Dew_Point: double (nullable = true)
 |-- Solar_Radiation: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)
 |-- Seasons: string (nullable = true)
 |-- Holiday: string (nullable = true)
 |-- Functioning_Day: string (nullable = true)
 |-- Month: double (nullable = true)
 |-- Day_of_Week: double (nullable = true)



+----------+----------+----+-----------+--------+----------+----------+---------+---------------+--------+--------+-------+----------+---------------+-----+-----------+
|      Date|Bike_Count|Hour|Temperature|Humidity|Wind_Speed|Visibility|Dew_Point|Solar_Radiation|Rainfall|Snowfall|Seasons|   Holiday|Functioning_Day|Month|Day_of_Week|
+----------+----------+----+-----------+--------+----------+----------+---------+---------------+--------+--------+-------+----------+---------------+-----+-----------+
|01/12/2017|     254.0| 0.0|       -5.2|    37.0|       2.2|    2000.0|    -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0|        6.0|
|01/12/2017|     204.0| 1.0|       -5.5|    38.0|       0.8|    2000.0|    -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0|        6.0|
|01/12/2017|     173.0| 2.0|       -6.0|    39.0|       1.0|    2000.0|    -17.7|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0

## Splitting the Data, Metrics, and Models

• Using spark MLlib, I split the data into a training and test set with an 80/20 split.

• I've chosen to use RMSE as the metric to judge my models.

• The three different classes of models I'll be fitting are: 

    1.Linear Regression with Elastic Net:  Elastic Net linear regression is a regularized method that combines L1 (Lasso) and L2 (Ridge) penalties to overcome limitations of both. It performs feature selection (driving some coefficients to zero) while keeping correlated predictors together. This works well when multiple features (variables) are correlated. Since environmental factors like temperature and humidity are often correlated, I thtink this is a promising model.
    
    2.Random Forest Regressor: Random forest is an ML algorithm that combines the output of multiple decision trees to reach a single result, crreating a more stable and accurate model.  It uses bootstrapping: Each tree is trained on a random sample of the training data, allowing for different data subsets. 
    
    3. Generalized Linear Regressor (Poisson): Regular linear regression assumes the response variable is continuous and normally distributed. But Rented Bike Count is a count that is always a whole number and can't be negative. Count data often follows a Poisson distribution instead of a normal one. Poisson regression handles this by modeling the log of the expected count as a linear combination of the predictors, which also guarantees predictions are always positive.

In [8]:
# split into training and test roughly 80/20
train, test = sdf.randomSplit([0.8, 0.2], seed = 8)



Creating my pipeline transformations.

In [9]:
# StringIndexer to convert seasons and holidays to numeric
indexer = StringIndexer(
    inputCols=['Seasons', 'Holiday'],
    outputCols=['Seasons_idx', 'Holiday_idx'])

# OneHotEncoder to convert index to dummy
encoder =OneHotEncoder(
    inputCols=['Seasons_idx', 'Holiday_idx'],
    outputCols=['Seasons_enc', 'Holiday_enc'])

# SQLTransformer to log transform response
sqlTrans= SQLTransformer(statement = 
                         "SELECT *, log(Bike_Count) as label FROM __THIS__")

#VectorAssembler to bundle features together into one vector
assembler = VectorAssembler(
    inputCols=['Hour', 'Temperature', 'Humidity', 'Wind_Speed', 'Visibility',
               'Dew_Point', 'Solar_Radiation', 'Rainfall', 'Snowfall',
               'Month', 'Day_of_Week', 'Seasons_enc', 'Holiday_enc'],
    outputCol='features')

print("transformations complete")

transformations complete


## Model Fitting
I used Spark MLlib to fit my three different classes models to the training data. This
was done using pipelines and cross validation to choose the best model for each model type. I
compared my models using RMSE as the metric.

Each model got it's own pipeline.  The transformations were done using the functions from MLlib to easily put them into the pipeline.


### Linear Regression Model with elastic net

In [10]:
from pyspark.ml import Pipeline

# define lr model
lr = LinearRegression()

paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.01, 0.05, 0.1]) \
    .addGrid(lr.elasticNetParam, [0, 0.5, 1.0]) \
    .build()

#build lr pipeline
pipeline_lr= Pipeline(stages = [indexer, encoder, sqlTrans, assembler, lr])

crossval_lr = CrossValidator(estimator = pipeline_lr,
                          estimatorParamMaps = paramGrid_lr,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)
#fit trraining data
cvModel_lr = crossval_lr.fit(train)
print("LR done")

26/04/14 08:23:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/14 08:23:59 WARN Instrumentation: [9fbaf1b1] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:02 WARN Instrumentation: [aa2e38a3] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:03 WARN Instrumentation: [ea5d9378] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:14 WARN Instrumentation: [6b334794] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:15 WARN Instrumentation: [598bd299] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:16 WARN Instrumentation: [14d7eab6] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:24:23 WARN Instrumentation: [462cce78] regP

LR done


Now we'll check which parameters were best. From below it is regParm=0, elasticNetParm=0 with an RMSE=0.713. Including all the coefficients with no shrinking was best.

In [11]:
#using professor's code 

my_list = []
for i in range(len(paramGrid_lr)):
    my_list.append([cvModel_lr.avgMetrics[i], paramGrid_lr[i].values()])
my_list

[[0.7123470577467426, dict_values([0.0, 0.0])],
 [0.712347057746742, dict_values([0.0, 0.5])],
 [0.7123470577467423, dict_values([0.0, 1.0])],
 [0.7146074307996934, dict_values([0.01, 0.0])],
 [0.7153393893739937, dict_values([0.01, 0.5])],
 [0.7166324781742937, dict_values([0.01, 1.0])],
 [0.7176907798261798, dict_values([0.05, 0.0])],
 [0.7251488816145839, dict_values([0.05, 0.5])],
 [0.7331673401745267, dict_values([0.05, 1.0])],
 [0.7207930267340955, dict_values([0.1, 0.0])],
 [0.7367927595136653, dict_values([0.1, 0.5])],
 [0.7591852697454222, dict_values([0.1, 1.0])]]

### Random Forest Model

In [12]:
## Defining random forest model
rf = RandomForestRegressor(seed=8)

paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [3, 5]) \
    .build()
    
# rf pipeline 
pipeline_rf= Pipeline(stages = [indexer, encoder, sqlTrans, assembler, rf])


crossval_rf = CrossValidator(estimator = pipeline_rf,
                          estimatorParamMaps = paramGrid_rf,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

#fit trraining data
cvModel_rf = crossval_rf.fit(train)

print("RF done")

RF done


We see below that the best RF model has numTrees=20, maxDepth=5 and RMSE=0.561.

In [13]:
my_list = []
for i in range(len(paramGrid_rf)):
    my_list.append([cvModel_rf.avgMetrics[i], paramGrid_rf[i].values()])
my_list

[[0.6815765077654139, dict_values([10, 3])],
 [0.5670965311775022, dict_values([10, 5])],
 [0.6791304946053531, dict_values([20, 3])],
 [0.5623796715852765, dict_values([20, 5])]]

### Generalized Linear Regression Poisson Model

In [14]:
## Defining GLM Poisson model
glr = GeneralizedLinearRegression(family='poisson', link='log')

paramGrid_glr = ParamGridBuilder() \
    .addGrid(glr.regParam, [0, 0.01, 0.05, 0.1]) \
    .build()
    
# glr pipeline 
pipeline_glr= Pipeline(stages = [indexer, encoder, sqlTrans, assembler, glr])


crossval_glr = CrossValidator(estimator = pipeline_glr,
                          estimatorParamMaps = paramGrid_glr,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

#fit trraining data
cvModel_glr = crossval_glr.fit(train)

print("GLR done")

26/04/14 08:25:27 WARN Instrumentation: [f8e73e8f] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:27 WARN Instrumentation: [f8e73e8f] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:27 WARN Instrumentation: [f8e73e8f] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:27 WARN Instrumentation: [f8e73e8f] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:31 WARN Instrumentation: [6c9138e9] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:31 WARN Instrumentation: [6c9138e9] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:32 WARN Instrumentation: [6c9138e9] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 08:25:32 WARN Instrumentation: [6c9138e9] regParam is zero, which might cause numerical instability and overf

GLR done


The best GLR model is 0.0 so plain Poisson regression.

In [15]:
my_list = []
for i in range(len(paramGrid_glr)):
    my_list.append([cvModel_glr.avgMetrics[i], paramGrid_glr[i].values()])
my_list

[[0.7043616675101757, dict_values([0.0])],
 [0.7057814900365569, dict_values([0.01])],
 [0.713922375990269, dict_values([0.05])],
 [0.7214544973330811, dict_values([0.1])]]

## Model Testing
Finally we can evaluate the models that were best in each category on the test set.

In [16]:

# test rmse's

test_rmse_lr = RegressionEvaluator(metricName='rmse').evaluate(cvModel_lr.transform(test))
test_rmse_rf = RegressionEvaluator(metricName='rmse').evaluate(cvModel_rf.transform(test))
test_rmse_glr = RegressionEvaluator(metricName='rmse').evaluate(cvModel_glr.transform(test))

print(f"Linear Regression Test RMSE = {test_rmse_lr}")
print(f"Random Forest Test RMSE     = {test_rmse_rf}")
print(f"Poisson GLR Test RMSE       = {test_rmse_glr}")


Linear Regression Test RMSE = 0.7962709439183023
Random Forest Test RMSE     = 0.5865431145430103
Poisson GLR Test RMSE       = 0.7492263829171397


The model that had the lowest RMSE was the Random Forest model with numTrees=20 and maxDepth=5 and an RMSE of 0.587